# Shot Analyzer Demo

End-to-end walkthrough of the shot-analysis pipeline on a sample cricket clip.

**Pipeline stages**
1. Run existing YOLO ball detector + tracker over the clip
2. (One-time) calibrate the ground-plane homography by clicking 4 pitch corners
3. Run `analyse_delivery` to get the contact event, 2 s prediction, angle, and shot name
4. Render the annotated output video

**Before running:** point `BALL_MODEL_PATH`, `BAT_MODEL_PATH`, and `VIDEO_PATH` at files that exist locally.
If you don't yet have a bat detector, see the *No bat model?* fallback cell below.

In [ ]:
import os, json
import numpy as np
import cv2

from shot_analyzer import (
    build_tracks_from_video,
    calibrate_from_clicks,
    save_homography,
    load_homography,
    analyse_delivery,
    render_overlay,
)

# ---- EDIT THESE PATHS ----
BALL_MODEL_PATH = r"ball_test/weights/best.pt"
BAT_MODEL_PATH  = r"C:\Cricket-Angle\models\yolo\best.pt"  # set to your bat YOLO weights, or leave None and use the fallback cell below
VIDEO_PATH      = r"C:\Cricket-Angle\videoplayback.mp4"
OUTPUT_VIDEO    = r"outputs/videos/shot_overlay.mp4"
HOMOGRAPHY_JSON = r"outputs/calibration_H.json"

HANDEDNESS = "right"  # or "left"

os.makedirs("outputs/videos", exist_ok=True)

## 1. Run detector + tracker

This reuses the existing modules under `src/` — we do **not** re-implement detection.

In [ ]:
ball_track, ball_boxes, bat_boxes, fps = build_tracks_from_video(
    VIDEO_PATH,
    ball_model_path=BALL_MODEL_PATH,
    bat_model_path=BAT_MODEL_PATH,    # set to None if no bat model available
    ball_class_id=0,
    bat_class_id=0,                   # change if your bat class is not 0
    ball_conf=0.2,
    bat_conf=0.25,
    include_ball_boxes=True,
)

print(f"fps={fps:.1f}")
print(f"ball detections: {len(ball_track)} frames")
print(f"ball boxes     : {len(ball_boxes)} frames")
print(f"bat  detections: {len(bat_boxes)} frames")

### No bat model? Stand-in fallback

If you don't yet have a bat detector, you can hand-label the bat in a few frames (or use a fixed approximate bbox where the batsman stands) to test the rest of the pipeline. Skip this cell if `bat_boxes` is already populated.

In [ ]:
if not bat_boxes:
    # FALLBACK: assume the batsman stands roughly in the lower-centre of the frame.
    # Replace these numbers with actual bat positions for your clip.
    cap = cv2.VideoCapture(VIDEO_PATH)
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    cx, cy = w // 2, int(h * 0.72)
    bat_boxes = {f: (cx - 25, cy - 80, cx + 25, cy + 20) for f in range(n)}
    print(f"Stand-in bat_boxes installed for {len(bat_boxes)} frames at ({cx},{cy})")
else:
    print("bat_boxes already populated — skipping fallback.")

## 2. Calibrate the ground-plane homography (one-time per camera setup)

An OpenCV window will open showing the first frame. Click 4 pitch corners in this order:

1. **Striker-end off-side corner**
2. **Striker-end leg-side corner**
3. **Bowler-end leg-side corner**
4. **Bowler-end off-side corner**

Then press any key to confirm. Press `r` to reset, `q` to abort.

If you already calibrated and saved `outputs/calibration_H.json`, skip to the *Load saved H* cell.

In [ ]:
H = calibrate_from_clicks(VIDEO_PATH, save_path=HOMOGRAPHY_JSON)
print("Homography saved to", HOMOGRAPHY_JSON)
print(H)

### Load saved H (skip the calibration cell next time)

In [ ]:
# H = load_homography(HOMOGRAPHY_JSON)
# print(H)

## 3. Run the full analysis

In [ ]:
result = analyse_delivery(
    ball_track,
    bat_boxes,
    H,
    fps=fps,
    handedness=HANDEDNESS,
    # --- knobs (override as needed) ---
    proximity_px=30,
    angle_change_deg=25,
    contact_window=3,
    post_contact_frames=60,
    direction_lookahead=10,
)

print(json.dumps(result.to_dict(), indent=2)[:2000])

## 4. Render the annotated video

In [ ]:
if result.contact is not None:
    out = render_overlay(
        VIDEO_PATH,
        OUTPUT_VIDEO,
        ball_track,
        bat_boxes,
        result,
        H,
        ball_boxes=ball_boxes,
    )
    print("Overlay written to:", out)
else:
    print("No contact detected -> nothing to render. Reason:", result.reason)

## 5. Inspect the result

Quick textual summary.

In [ ]:
if result.contact:
    print(f"Contact frame    : {result.contact.frame_idx}")
    print(f"Contact proximity: {result.contact.proximity_px:.1f} px")
    print(f"Angle change     : {result.contact.angle_change_deg:.1f} deg")
    print(f"Shot angle       : {result.angle_deg:.1f} deg (clockwise from +Y/toward bowler)")
    print(f"Shot             : {result.shot_name}")
    print(f"Observed post-contact ground points: {len(result.observed_ground)}")
    print(f"Predicted ground points (2s @ 0.05s): {len(result.predicted_ground)}")
else:
    print("No contact:", result.reason)

## (Optional) Plot the ground trajectory

Wagon-wheel-style top-down view of the observed + predicted ground trajectory.

In [ ]:
try:
    import matplotlib.pyplot as plt

    if result.observed_ground or result.predicted_ground:
        fig, ax = plt.subplots(figsize=(6, 6))
        if result.observed_ground:
            xs = [p[1] for p in result.observed_ground]
            ys = [p[2] for p in result.observed_ground]
            ax.plot(xs, ys, 'o-', label='observed', markersize=4)
        if result.predicted_ground:
            xs = [p[1] for p in result.predicted_ground]
            ys = [p[2] for p in result.predicted_ground]
            ax.plot(xs, ys, '--', label='predicted (2s)')
        ax.plot(0, 0, 'k^', markersize=12, label='striker (origin)')
        ax.axhline(20.12, color='gray', linestyle=':', label='bowler crease (~20.12m)')
        ax.set_xlabel('X (m)  -- +X = off side (RH)')
        ax.set_ylabel('Y (m)  -- +Y = toward bowler')
        ax.set_aspect('equal')
        ax.grid(True, alpha=0.3)
        ax.legend()
        ax.set_title(f"Shot: {result.shot_name}  ({result.angle_deg:.1f} deg)")
        plt.show()
    else:
        print('No ground trajectory available to plot.')
except ImportError:
    print('matplotlib not installed — skipping plot.')